In [0]:
# =============================================================================
# NB17 — Anion-aware composition descriptors (X_material_v2) + band gap
#        consolidation + external data with a physically matched domain
# =============================================================================
# Why this notebook exists
#   NB16 found that a featuriser computed on the FULL formula reaches band-gap
#   R² ≈ 0.42–0.45 under DOI-grouped CV with composition purging, versus 0.14
#   for X_material under the identical protocol (NB15b). Root cause: NB02b
#   strips anions (O, N, S, F, ...) before computing descriptors, so X_material
#   cannot distinguish an oxide (LaTaO4, ~4 eV) from an oxynitride (LaTaON2,
#   ~2 eV). Anion chemistry is the dominant physical control of Eg (N 2p / S 3p
#   states raise the valence band). X_material was used in ALL activity
#   analyses (NB08–NB14), which therefore must be re-tested (NB18).
#
# Sections
#   A. Build X_material_v2 for every master row (anion-aware, full formula)
#      and write it to Delta for reuse.
#   B. Band gap, DOI-grouped 5×5 CV with composition purging (as NB15b):
#        OLD_XMATERIAL (NB02b, cation-only)  vs  V2_CATION_ONLY  vs  V2_ALL
#      → isolates the effect of including anions with the SAME featuriser.
#      y-randomization (V2_ALL, N_YRAND shuffles) and cluster-bootstrap CIs.
#   C. External band gaps (matbench_expt_gap) under three domains:
#        F_ALL       — all 4,604 entries (NB16: zero-shot R² −5.8)
#        F_SEMI      — Eg > 0 (metals excluded; project samples are all
#                      semiconductors by construction)
#        F_SEMI_ON   — Eg > 0 and contains O or N (oxide / oxynitride domain)
#      These restrictions are justified physically but decided AFTER NB16;
#      all three are reported.
#
# Inputs:  master, atom_props_curated, X_material, external file
#          (cached by NB16 at /dbfs/FileStore/matbench_expt_gap.json.gz)
# Outputs: ml_perovskites.X_material_v2, ml_perovskites.bandgap_nb17,
#          /tmp/nb17_bandgap.png
# SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import glob, gzip, hashlib, json, os, re, subprocess, sys, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute       import SimpleImputer
from sklearn.ensemble     import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics      import r2_score, mean_absolute_error
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS, N_BOOT, N_YRAND = 42, "ml_perovskites", 5, 5, 1000, 10
ANIONS = {"O", "N", "S", "Se", "Te", "F", "Cl", "Br", "I", "H"}
HALIDES = {"F", "Cl", "Br", "I"}

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 0. Parser + featuriser (identical parser to NB16)
# ---------------------------------------------------------------------------
ap = spark.table(f"{SCHEMA}.atom_props_curated").toPandas().set_index("Element")
for c in ap.columns:
    ap[c] = pd.to_numeric(ap[c], errors="coerce").astype(float)
PROPS = [c for c in ap.columns if ap[c].notna().mean() > 0.5]
ELEMENTS = set(ap.index)
TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)|(\()|(\))(\d*\.?\d*)")

def clean_formula(s):
    s = str(s)
    s = re.sub(r"\([^)]*\s[^)]*\)", "", s)
    s = re.sub(r"\([^)]*[a-z]{2,}[^)]*\)", "", s)
    return re.sub(r"\s+", "", s)

def parse(s):
    s = clean_formula(s)
    if not s or "/" in s or "-" in s:
        return None
    stack, i = [{}], 0
    while i < len(s):
        m = TOKEN.match(s, i)
        if not m:
            return None
        if m.group(1):
            el, n = m.group(1), m.group(2)
            if el not in ELEMENTS:
                return None
            stack[-1][el] = stack[-1].get(el, 0.0) + (float(n) if n not in ("", ".") else 1.0)
        elif m.group(3):
            stack.append({})
        else:
            if len(stack) < 2:
                return None
            mult = float(m.group(5)) if m.group(5) not in ("", ".") else 1.0
            top = stack.pop()
            for k, v in top.items():
                stack[-1][k] = stack[-1].get(k, 0.0) + v * mult
        i = m.end()
    if len(stack) != 1 or not stack[0] or sum(stack[0].values()) <= 0:
        return None
    tot = sum(stack[0].values())
    return {k: v / tot for k, v in stack[0].items()}

def comp_key(fr):
    return "".join(f"{k}{v:.3f}" for k, v in sorted(fr.items()))

def stats(fr, prefix):
    out = {}
    els, x = list(fr), np.array(list(fr.values()))
    for p in PROPS:
        v = ap.loc[els, p].values
        ok = ~np.isnan(v)
        if not ok.any():
            for s in ["mean", "std", "min", "max", "range"]:
                out[f"{prefix}{p}_{s}"] = np.nan
            continue
        w = x[ok] / x[ok].sum()
        mu = float(np.sum(w * v[ok]))
        out[f"{prefix}{p}_mean"] = mu
        out[f"{prefix}{p}_std"] = float(np.sqrt(np.sum(w * (v[ok] - mu) ** 2)))
        out[f"{prefix}{p}_min"] = float(v[ok].min())
        out[f"{prefix}{p}_max"] = float(v[ok].max())
        out[f"{prefix}{p}_range"] = float(v[ok].max() - v[ok].min())
    return out

def featurise_all(fr):
    """Anion-aware: statistics over the full formula + explicit anion fractions."""
    out = {"v2_n_elements": len(fr)}
    out.update(stats(fr, "v2_"))
    for a in ["O", "N", "S", "H"]:
        out[f"v2_frac_{a}"] = fr.get(a, 0.0)
    out["v2_frac_halide"] = sum(fr.get(h, 0.0) for h in HALIDES)
    out["v2_frac_anion"] = sum(v for k, v in fr.items() if k in ANIONS)
    return out

def featurise_cation(fr):
    """Control: same statistics, anions removed (mimics NB02b's choice)."""
    cat = {k: v for k, v in fr.items() if k not in ANIONS}
    tot = sum(cat.values())
    if tot <= 0:
        return {}
    cat = {k: v / tot for k, v in cat.items()}
    out = {"c_n_elements": len(cat)}
    out.update(stats(cat, "c_"))
    return out

# ---------------------------------------------------------------------------
# A. X_material_v2 for every master row
# ---------------------------------------------------------------------------
master = spark.table(f"{SCHEMA}.master").toPandas()
master["source_row_id"] = master["source_row_id"].astype(str).str.strip()
fr_master = master["base_compound"].map(lambda s: parse(s) if pd.notna(s) else None)
ok_m = fr_master.notna()
Xv2 = pd.DataFrame([featurise_all(f) for f in fr_master[ok_m]], index=master.index[ok_m])
Xv2.insert(0, "source_row_id", master.loc[ok_m, "source_row_id"].values)
Xv2_full = master[["source_row_id"]].merge(Xv2, on="source_row_id", how="left")
print(f"X_material_v2: {ok_m.sum()}/{len(master)} rows parsed | {Xv2.shape[1] - 1} features")
print(f"  unparsed base_compound examples: "
      f"{master.loc[~ok_m, 'base_compound'].dropna().unique()[:8].tolist()}")
spark.createDataFrame(Xv2_full.replace({np.nan: None})).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.X_material_v2")
print(f"✓ Written: {SCHEMA}.X_material_v2 ({len(Xv2_full)} rows)")

# ---------------------------------------------------------------------------
# B. Band gap — OLD vs V2_CATION_ONLY vs V2_ALL (DOI-grouped, composition purged)
# ---------------------------------------------------------------------------
X_old_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_old_raw["source_row_id"] = X_old_raw["source_row_id"].astype(str).str.strip()
for c in X_old_raw.columns:
    if c != "source_row_id":
        X_old_raw[c] = pd.to_numeric(X_old_raw[c], errors="coerce")
META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key"}
OLD_FEATS = [c for c in X_old_raw.columns if c not in META
             and not re.match(r"^(ECB|EVB)_", c)
             and pd.api.types.is_numeric_dtype(X_old_raw[c])
             and X_old_raw[c].isna().mean() < 0.50]

master["eg"] = pd.to_numeric(master["bandgap_eV"], errors="coerce")
d = master[master["eg"].between(0.0, 6.5) & master["doi_key"].notna() &
           master["base_compound"].notna()].copy()
skey = "Compound_original" if "Compound_original" in d.columns else "base_compound"
d = (d.groupby(["doi_key", skey], as_index=False)
       .agg(source_row_id=("source_row_id", "first"), eg=("eg", "mean"),
            base_compound=("base_compound", "first")))
d["fr"] = d["base_compound"].map(parse)
d = d[d["fr"].notna()].reset_index(drop=True)
F_all = pd.DataFrame([featurise_all(f) for f in d["fr"]])
F_cat = pd.DataFrame([featurise_cation(f) for f in d["fr"]])
F_old = d[["source_row_id"]].merge(X_old_raw[["source_row_id"] + OLD_FEATS],
                                   on="source_row_id", how="left")[OLD_FEATS]
print(f"\nBand-gap samples: {len(d)} | DOIs {d['doi_key'].nunique()} | "
      f"features: old {len(OLD_FEATS)}, v2_all {F_all.shape[1]}, v2_cation {F_cat.shape[1]}")
anion_mix = pd.Series({a: int((F_all[f"v2_frac_{a}"] > 0).sum()) for a in ["O", "N", "S", "H"]})
print(f"  samples containing O / N / S / H: {anion_mix.to_dict()}")

y = d["eg"].values
dois = np.array(sorted(d["doi_key"].unique()))
comps = d["base_compound"].values

def rf():
    return RandomForestRegressor(n_estimators=500, min_samples_leaf=2, max_features=0.3,
                                 n_jobs=-1, random_state=SEED)

def xgbm(n=500):
    return xgb.XGBRegressor(n_estimators=n, learning_rate=0.05, max_depth=5, subsample=0.8,
                            colsample_bytree=0.6, min_child_weight=2, tree_method="hist",
                            random_state=SEED, verbosity=0)

def folds(rep):
    perm = np.random.default_rng(SEED + rep).permutation(dois)
    f_of = {g: i % N_FOLDS for i, g in enumerate(perm)}
    f = d["doi_key"].map(f_of).values
    out = []
    for k in range(N_FOLDS):
        va = np.where(f == k)[0]
        tr = np.where(f != k)[0]
        tr = tr[~np.isin(comps[tr], list(set(comps[va])))]
        out.append((tr, va))
    return out

FOLDS = {rep: folds(rep) for rep in range(N_REPEATS)}

def cv_predict(X, yy, rep, make):
    p = np.full(len(yy), np.nan)
    for tr, va in FOLDS[rep]:
        imp = SimpleImputer(strategy="median").fit(X[tr])
        p[va] = make().fit(imp.transform(X[tr]), yy[tr]).predict(imp.transform(X[va]))
    return p

SETS = {"OLD_XMATERIAL_RF": (F_old.values.astype(float), rf),
        "V2_CATION_ONLY_RF": (F_cat.values.astype(float), rf),
        "V2_ALL_RF": (F_all.values.astype(float), rf),
        "V2_ALL_XGB": (F_all.values.astype(float), xgbm)}
rows, oof0 = [], {}
for name, (X, make) in SETS.items():
    for rep in range(N_REPEATS):
        p = cv_predict(X, y, rep, make)
        rows.append({"block": "B", "config": name, "repeat": rep,
                     "r2": r2_score(y, p), "mae_eV": mean_absolute_error(y, p)})
        if rep == 0:
            oof0[name] = p
    print(f"  {name} done")

# y-randomization on V2_ALL_RF (repeat-0 folds)
rng = np.random.default_rng(SEED + 11)
yr_r2 = []
for _ in range(N_YRAND):
    yr = rng.permutation(y)
    yr_r2.append(r2_score(yr, cv_predict(F_all.values.astype(float), yr, 0, rf)))

# ---------------------------------------------------------------------------
# C. External data under three domains
# ---------------------------------------------------------------------------
SOURCES = {"783e7d1461eb83b00b2f2942da4b95fda5e58a0d1ae26b581c24cf8a82ca75b2": "matbench_expt_gap"}
EXT_URL = "https://ml.materialsproject.org/projects/matbench_expt_gap.json.gz"
raw = None
for p in ["/dbfs/FileStore/matbench_expt_gap.json.gz"] + \
         glob.glob("/Volumes/*/*/*/matbench_expt_gap.json.gz"):
    if os.path.exists(p):
        blob = open(p, "rb").read()
        if hashlib.sha256(blob).hexdigest() in SOURCES:
            raw = blob
            break
if raw is None:
    req = urllib.request.Request(EXT_URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=120).read()
    assert hashlib.sha256(raw).hexdigest() in SOURCES, "SHA-256 mismatch"
obj = json.loads(gzip.decompress(raw).decode("utf-8"))
ext = pd.DataFrame(obj["data"], columns=obj["columns"]) if isinstance(obj, dict) and "data" in obj \
    else pd.DataFrame(obj)
ccol = "composition" if "composition" in ext.columns else "formula"
ext = pd.DataFrame({"formula": ext[ccol].astype(str),
                    "eg": pd.to_numeric(ext["gap expt"], errors="coerce")}).dropna()
ext["fr"] = ext["formula"].map(parse)
ext = ext[ext["fr"].notna()].reset_index(drop=True)
ext["key"] = ext["fr"].map(comp_key)
proj_keys = set(d["fr"].map(comp_key))
ext = ext[~ext["key"].isin(proj_keys)].reset_index(drop=True)       # leakage purge
ext["has_ON"] = ext["fr"].map(lambda f: ("O" in f) or ("N" in f))
DOMAINS = {"F_ALL": np.ones(len(ext), bool),
           "F_SEMI": (ext["eg"] > 0).values,
           "F_SEMI_ON": ((ext["eg"] > 0) & ext["has_ON"]).values}
FX_ext = pd.DataFrame([featurise_all(f) for f in ext["fr"]])[F_all.columns].values.astype(float)
Xp = F_all.values.astype(float)
print(f"\nExternal after purge: {len(ext)} | domain sizes: "
      f"{ {k: int(v.sum()) for k, v in DOMAINS.items()} }")

for dname, mask in DOMAINS.items():
    Xe, ye = FX_ext[mask], ext["eg"].values[mask]
    imp_e = SimpleImputer(strategy="median").fit(Xe)
    zs = xgbm(600).fit(imp_e.transform(Xe), ye).predict(imp_e.transform(Xp))
    for rep in range(N_REPEATS):
        P = {k: np.full(len(y), np.nan) for k in ["ZERO_SHOT", "CALIBRATED", "STACK_RF", "COMBINED_XGB"]}
        for tr, va in FOLDS[rep]:
            P["ZERO_SHOT"][va] = zs[va]
            P["CALIBRATED"][va] = LinearRegression().fit(zs[tr, None], y[tr]).predict(zs[va, None])
            imp = SimpleImputer(strategy="median").fit(Xp[tr])
            P["STACK_RF"][va] = rf().fit(np.c_[imp.transform(Xp[tr]), zs[tr]], y[tr]) \
                                    .predict(np.c_[imp.transform(Xp[va]), zs[va]])
            P["COMBINED_XGB"][va] = xgbm(400).fit(
                np.r_[imp_e.transform(Xe), imp_e.transform(Xp[tr])], np.r_[ye, y[tr]]
            ).predict(imp_e.transform(Xp[va]))
        for k, p in P.items():
            name = f"{dname}_{k}"
            rows.append({"block": "C", "config": name, "repeat": rep,
                         "r2": r2_score(y, p), "mae_eV": mean_absolute_error(y, p)})
            if rep == 0:
                oof0[name] = p
    print(f"  domain {dname} done")

# ---------------------------------------------------------------------------
# Summary with cluster-bootstrap CI and paired ΔR² vs V2_ALL_RF and vs OLD
# ---------------------------------------------------------------------------
res = pd.DataFrame(rows)
summ = res.groupby(["block", "config"]).agg(r2=("r2", "mean"), r2_min=("r2", "min"),
                                            r2_max=("r2", "max"),
                                            mae_eV=("mae_eV", "mean")).reset_index()
by = {g: np.where(d["doi_key"].values == g)[0] for g in dois}
rng = np.random.default_rng(SEED + 3)
boot = [np.concatenate([by[g] for g in rng.choice(dois, len(dois), replace=True)])
        for _ in range(N_BOOT)]
def r2b(p):
    return np.array([r2_score(y[i], p[i]) for i in boot])
base_old, base_v2 = r2b(oof0["OLD_XMATERIAL_RF"]), r2b(oof0["V2_ALL_RF"])
extra = []
for c in summ["config"]:
    b = r2b(oof0[c])
    extra.append({"config": c, "ci_lo": np.percentile(b, 2.5), "ci_hi": np.percentile(b, 97.5),
                  "dR2_vs_old": np.mean(b - base_old),
                  "dR2_vs_old_lo": np.percentile(b - base_old, 2.5),
                  "dR2_vs_v2": np.mean(b - base_v2),
                  "dR2_vs_v2_lo": np.percentile(b - base_v2, 2.5),
                  "dR2_vs_v2_hi": np.percentile(b - base_v2, 97.5)})
summ = summ.merge(pd.DataFrame(extra), on="config")
pd.set_option("display.width", 240)
print(f"\n--- Band gap, DOI-grouped CV with composition purging ({N_REPEATS}×{N_FOLDS}); "
      f"CI / ΔR² = cluster bootstrap by DOI (repeat 0) ---")
print(summ.round(3).to_string(index=False))
print(f"\ny-randomization V2_ALL_RF ({N_YRAND} shuffles): R² mean {np.mean(yr_r2):+.3f}, "
      f"max {np.max(yr_r2):+.3f}")
print("\nReading guide:")
print("  * V2_ALL vs V2_CATION_ONLY (same featuriser) isolates the effect of anions.")
print("  * External domains: ΔR² vs V2_ALL with CI above 0 → external data helps.")

# Plot: parity OLD vs V2_ALL vs best external
best_ext = summ[summ["block"] == "C"].sort_values("r2", ascending=False)["config"].iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
lim = (max(0, y.min() - 0.2), y.max() + 0.2)
for ax, c in zip(axes, ["OLD_XMATERIAL_RF", "V2_ALL_RF", best_ext]):
    s = summ[summ["config"] == c].iloc[0]
    ax.scatter(y, oof0[c], s=12, alpha=0.6)
    ax.plot(lim, lim, color="black", lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Measured Eg (eV)"); ax.set_ylabel("Predicted Eg (eV)")
    ax.set_title(f"{c}\nR² {s['r2']:.2f} [{s['ci_lo']:.2f}, {s['ci_hi']:.2f}]", fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/nb17_bandgap.png", dpi=130); plt.close()
print("\nSaved: /tmp/nb17_bandgap.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.bandgap_nb17")
print(f"✓ Written: {SCHEMA}.bandgap_nb17 ({len(res)} rows)")
print("\n=== NB17 COMPLETE ===")
print("Next: NB18 — re-run activity tests (NB12, NB14) with X_material_v2.")